# Tutorial de AutoCDC: Dejá de Codear a Mano Pipelines de Change Data Capture

## Introducción

Bienvenido a este tutorial hands-on sobre **AutoCDC** - el approach declarativo de Databricks para automatizar pipelines de Change Data Capture (CDC) y Slowly Changing Dimensions (SCD).

### ¿Qué es AutoCDC?

AutoCDC es una feature de Delta Live Tables que maneja automáticamente:
- **SCD Type 1**: Sobreescribir con últimos valores (sin historia)
- **SCD Type 2**: Trackear historia completa de cambios con versionado
- **Snapshot-based CDC**: Derivar cambios desde snapshots periódicos

### ¿Por qué usar AutoCDC?

Los pipelines CDC tradicionales requieren lógica compleja hecha a mano:
- Statements MERGE custom
- Manejo manual de evolución de schema
- Gestión de estado compleja
- Lógica de versionado propensa a errores

**AutoCDC elimina todo esto** con sintaxis declarativa que es:
- ✅ Simple y mantenible
- ✅ Maneja automáticamente cambios de schema
- ✅ Hasta 2x mejor price-performance
- ✅ Optimización built-in

### Sobre los datos de muestra

Este tutorial usa:
- **Origen**: El **catálogo de muestras Delta Shared** (`samples.tpch`) - disponible en todos los workspaces de Databricks
- **Destino**: Tu **catálogo hablando_de_data** con el schema `default`

**TPC-H** es un benchmark estándar de decision support que contiene datos de negocio realistas:
- **customer**: Demografía de clientes e info de cuentas (vamos a usar esta tabla)
- **orders**: Órdenes de compra con fechas y totales
- **lineitem**: Line items individuales de órdenes
- **part**: Catálogo de partes con precios
- **supplier**: Información de suppliers
- **nation/region**: Datos de referencia geográfica

La tabla **customer** representa una dimensión típica que cambia en el tiempo (actualizaciones de dirección, cambios de teléfono, balances de cuenta).

### Estructura del tutorial (simplificada)

Este tutorial simplificado sigue la **Arquitectura Medallion**:
1. **Capa Bronze**: Ingesta de datos crudos con metadata CDC
2. **Capa Silver**: Aplicar transformación SCD Type 1 (solo últimos valores)
3. **Capa Gold**: Métricas de negocio agregadas

Todas las tablas se van a crear en tu schema **`hablando_de_data.default`**.

---

## Setup: Ambiente y Datos de Muestra

Vamos a usar el **catálogo de muestras Delta Shared**, que está disponible en todos los workspaces de Databricks. Esto asegura que cualquiera pueda correr este tutorial sin setup adicional.

In [0]:
# Importar librerías necesarias
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timedelta
import random

# Configurar catálogo y database
catalog = "samples"
database = "tpch"

print(f"✅ Usando catálogo: {catalog}")
print(f"✅ Usando database: {database}")
print(f"\n📊 Tablas de muestra disponibles en {catalog}.{database}:")

# Listar tablas disponibles
tables = spark.sql(f"SHOW TABLES IN {catalog}.{database}").select("tableName").collect()
for table in tables:
    print(f"   - {table.tableName}")

## Parte 1: Entender los Datos de Origen

Exploremos la tabla **customer** de TPC-H, que vamos a usar para nuestros ejemplos de CDC.

In [0]:
# Explorar el schema de la tabla customer
customer_df = spark.table(f"{catalog}.{database}.customer")

print("📋 Schema de la Tabla Customer:")
customer_df.printSchema()

print(f"\n📊 Total de Registros: {customer_df.count():,}")

print("\n🔍 Registros de Muestra:")
display(customer_df.limit(10))

## Parte 2: Simular Streams de Datos CDC

Para este tutorial, vamos a simular eventos CDC creando:
1. **Snapshot inicial**: Estado inicial de clientes
2. **Eventos de update**: Cambios a registros de clientes existentes
3. **Eventos de insert**: Clientes nuevos

En un escenario real, estos datos vendrían de herramientas CDC como Debezium, Oracle GoldenGate, o transaction logs de bases de datos.

In [0]:
# Crear un schema en el catálogo hablando_de_data para nuestro tutorial
catalog_name = "hablando_de_data"
schema_name = "default"
work_schema = f"{catalog_name}.{schema_name}"

# Usar el catálogo
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

print(f"✅ Usando catálogo: {catalog_name}")
print(f"✅ Usando schema: {work_schema}")

### Capa Bronze: Ingestar Eventos CDC Crudos

Siguiendo la arquitectura Medallion, primero creamos una **tabla Bronze** con eventos CDC crudos.

In [0]:
# Tomar una muestra de clientes y agregar metadata CDC
# Esto simula un snapshot inicial
bronze_customers = (
    customer_df
    .limit(100)  # Trabajar con 100 clientes para la demo
    .withColumn("operation", F.lit("INSERT"))  # Tipo de operación CDC
    .withColumn("change_timestamp", F.lit(datetime(2026, 3, 1, 10, 0, 0)))  # Cuándo ocurrió el cambio
    .withColumn("sequence_number", F.monotonically_increasing_id())  # Orden de cambios
)

# Escribir a la capa Bronze
bronze_table = f"{work_schema}.bronze_customer_cdc"
bronze_customers.write.format("delta").mode("overwrite").saveAsTable(bronze_table)

print(f"✅ Tabla Bronze creada: {bronze_table}")
print(f"📊 Registros iniciales: {spark.table(bronze_table).count():,}")

display(spark.table(bronze_table).limit(5))

### Simular Actualizaciones de Clientes (Eventos CDC)

Simulemos algunos cambios en los datos de clientes:
- Actualizar números de teléfono
- Actualizar balances de cuenta
- Actualizar segmentos de mercado

In [0]:
from decimal import Decimal

# Obtener algunos clientes existentes para actualizar
existing_customers = spark.table(bronze_table).limit(20).collect()

# Crear eventos de update
update_records = []
for i, customer in enumerate(existing_customers[:10]):
    # Crear una versión actualizada
    update_records.append((
        customer.c_custkey,
        customer.c_name,
        customer.c_address,
        customer.c_nationkey,
        f"##-###-###-{random.randint(1000, 9999)}",  # Teléfono nuevo
        Decimal(str(round(float(customer.c_acctbal) + random.uniform(-1000, 1000), 2))),  # Balance actualizado
        customer.c_mktsegment,
        customer.c_comment,
        "UPDATE",  # Tipo de operación
        datetime(2026, 3, 15, 14, 30, 0),  # Timestamp del cambio
        1000 + i  # Número de secuencia
    ))

# Crear eventos de insert (clientes nuevos)
for i in range(5):
    new_id = 200000 + i
    update_records.append((
        new_id,
        f"Customer_{new_id}",
        f"Address {new_id}",
        random.randint(0, 24),
        f"##-###-###-{random.randint(1000, 9999)}",
        Decimal(str(round(random.uniform(0, 10000), 2))),
        random.choice(["AUTOMOBILE", "BUILDING", "FURNITURE", "MACHINERY", "HOUSEHOLD"]),
        "Cliente nuevo",
        "INSERT",
        datetime(2026, 3, 20, 9, 0, 0),
        2000 + i
    ))

# Crear DataFrame con las actualizaciones
update_schema = bronze_customers.schema
updates_df = spark.createDataFrame(update_records, schema=update_schema)

# Appendear a la capa Bronze
updates_df.write.format("delta").mode("append").saveAsTable(bronze_table)

print(f"✅ Eventos CDC agregados a la tabla Bronze")
print(f"📊 Total de registros ahora: {spark.table(bronze_table).count():,}")
print(f"\n🔄 Resumen de Operaciones CDC:")
display(spark.table(bronze_table).groupBy("operation").count().orderBy("operation"))

## Parte 3: SCD Type 1 - Solo Últimos Valores

**SCD Type 1** mantiene solo el estado actual sobreescribiendo valores viejos con nuevos. No se preserva historia.

### Casos de uso:
- Corregir errores de datos
- Atributos dimensionales no críticos
- Cuando no se requiere historia

### Cómo AutoCDC simplifica esto:

**Sin AutoCDC** (approach manual):
```python
# Lógica MERGE compleja
target.alias("t").merge(
    source.alias("s"),
    "t.id = s.id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()
```

**Con AutoCDC en Delta Live Tables**:
```python
@dlt.table
@dlt.apply_changes(
    target="customers_scd1",
    source="bronze_customer_cdc",
    keys=["c_custkey"],
    sequence_by="change_timestamp",
    stored_as_scd_type="1"
)
```

¡Eso es todo! AutoCDC maneja toda la complejidad.

### Capa Silver: Aplicar SCD Type 1 (Implementación Manual)

Como estamos en un notebook estándar (no DLT), implementemos SCD Type 1 manualmente para entender qué hace AutoCDC por vos:

In [0]:
from delta.tables import DeltaTable

# Tabla Silver para SCD Type 1
silver_scd1_table = f"{work_schema}.silver_customer_scd1"

# Obtener el estado más reciente de cada cliente (por change_timestamp)
window_spec = Window.partitionBy("c_custkey").orderBy(F.col("change_timestamp").desc())

latest_customers = (
    spark.table(bronze_table)
    .withColumn("row_num", F.row_number().over(window_spec))
    .filter(F.col("row_num") == 1)
    .drop("row_num", "operation", "sequence_number")
)

# Escribir a la tabla Silver SCD Type 1
latest_customers.write.format("delta").mode("overwrite").saveAsTable(silver_scd1_table)

print(f"✅ Tabla Silver SCD Type 1 creada: {silver_scd1_table}")
print(f"📊 Cantidad de clientes actuales: {spark.table(silver_scd1_table).count():,}")
print("\n🔍 Registros de muestra (mostrando solo estado latest):")
display(spark.table(silver_scd1_table).orderBy("c_custkey").limit(10))

### Verificar SCD Type 1: Sin Historia Preservada

Verifiquemos que solo tenemos la última versión de cada cliente:

In [0]:
# Verificar un cliente específico que fue actualizado
sample_custkey = existing_customers[0].c_custkey

print(f"🔍 Verificando cliente: {sample_custkey}")
print("\n📋 En Bronze (todos los eventos CDC):")
display(
    spark.table(bronze_table)
    .filter(F.col("c_custkey") == sample_custkey)
    .orderBy("change_timestamp")
)

print("\n📋 En Silver SCD Type 1 (solo latest):")
display(
    spark.table(silver_scd1_table)
    .filter(F.col("c_custkey") == sample_custkey)
)

## Parte 5: Capa Gold - Métricas de Negocio

Siguiendo la arquitectura Medallion, la **capa Gold** contiene métricas de negocio agregadas listas para analítica y herramientas de BI.

Creemos un resumen de balances de cuenta de clientes por segmento de mercado:

### Métrica Gold 1: Resumen de Balance de Cuenta de Clientes

In [0]:
# Crear métricas agregadas desde Silver SCD Type 1
gold_balance_summary = f"{work_schema}.gold_customer_balance_summary"

balance_metrics = (
    spark.table(silver_scd1_table)
    .groupBy("c_mktsegment")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum("c_acctbal").alias("total_balance"),
        F.avg("c_acctbal").alias("avg_balance"),
        F.min("c_acctbal").alias("min_balance"),
        F.max("c_acctbal").alias("max_balance")
    )
    .orderBy(F.desc("total_balance"))
)

balance_metrics.write.format("delta").mode("overwrite").saveAsTable(gold_balance_summary)

print(f"✅ Métrica Gold creada: {gold_balance_summary}")
print("\n💰 Balance de Clientes por Segmento de Mercado:")
display(spark.table(gold_balance_summary))

## Usar AutoCDC en Delta Live Tables (Producción)

Para usar AutoCDC en producción, crearías un **pipeline de Delta Live Tables**. Acá está el código completo siguiendo la arquitectura Medallion:

```python
import dlt
from pyspark.sql import functions as F

# Bronze: Ingestar datos CDC crudos
@dlt.table(
    comment="Eventos CDC crudos desde el sistema origen",
    table_properties={"quality": "bronze"}
)
def bronze_customer_cdc():
    return (
        spark.readStream
        .format("delta")
        .table("source_cdc_stream")
    )

# Silver: Aplicar SCD Type 1 con AutoCDC
dlt.create_streaming_table(
    name="silver_customer_scd1",
    comment="Dimensión de cliente con últimos valores (SCD Type 1)",
    table_properties={"quality": "silver"}
)

dlt.apply_changes(
    target="silver_customer_scd1",
    source="bronze_customer_cdc",
    keys=["c_custkey"],                    # Primary key
    sequence_by="change_timestamp",         # Orden de cambios
    stored_as_scd_type="1",                 # SCD Type 1
    except_column_list=["operation"]        # Excluir metadata CDC
)

# Gold: Métricas de negocio agregadas
@dlt.table(
    comment="Resumen de balance de clientes por segmento de mercado",
    table_properties={"quality": "gold"}
)
def gold_customer_balance_summary():
    return (
        dlt.read("silver_customer_scd1")
        .groupBy("c_mktsegment")
        .agg(
            F.count("*").alias("customer_count"),
            F.sum("c_acctbal").alias("total_balance"),
            F.avg("c_acctbal").alias("avg_balance")
        )
    )
```

### Parámetros clave de AutoCDC:

* **target**: Nombre de la tabla destino
* **source**: Tabla streaming origen con eventos CDC
* **keys**: Columnas de primary key para identificar registros
* **sequence_by**: Columna para determinar orden de cambios
* **stored_as_scd_type**: "1" para últimos valores, "2" para historia completa
* **except_column_list**: Columnas a excluir de la tabla destino

### Para SCD Type 2 (Historia Completa):

Simplemente cambiá `stored_as_scd_type="2"` y AutoCDC automáticamente:
- Agregará columnas de timestamp `__START_AT` y `__END_AT`
- Trackeará todas las versiones históricas
- Mantendrá indicadores de registro actual

¡Ese es el poder de AutoCDC - un cambio de parámetro maneja toda la complejidad!

## Parte 9: Limpieza (Opcional)

Si querés eliminar las tablas del tutorial:

In [0]:
# Descomentar para limpiar las tablas del tutorial
# spark.sql(f"DROP TABLE IF EXISTS {bronze_table}")
# spark.sql(f"DROP TABLE IF EXISTS {silver_scd1_table}")
# spark.sql(f"DROP TABLE IF EXISTS {gold_balance_summary}")
# print(f"✅ Tablas del tutorial eliminadas de {work_schema}")

print("ℹ️ Para limpiar, descomentar el código arriba y ejecutar esta celda")

## Resumen y Conclusiones Clave

### Qué aprendimos:

1. **AutoCDC elimina lógica CDC compleja**
   - Sin statements MERGE manuales
   - Sin lógica de versionado compleja
   - Sintaxis declarativa con `dlt.apply_changes()`

2. **Arquitectura Medallion en la Práctica**
   - **Bronze**: Eventos CDC crudos con metadata de operación
   - **Silver**: Dimensión limpia usando SCD Type 1
   - **Gold**: Métricas de negocio agregadas

3. **Patrones SCD**
   - **Type 1**: Solo últimos valores (sobrescribe) - mostrado en este tutorial
   - **Type 2**: Trackeo de historia completa - ¡solo cambiás un parámetro!

4. **Beneficios de Performance**
   - Hasta 2x mejor price-performance
   - Optimización automática
   - Evolución de schema built-in

### Próximos pasos:

1. **Probá AutoCDC con tus datos**
   - Identificá tablas que necesiten CDC
   - Creá un pipeline de Delta Live Tables
   - Usá `dlt.apply_changes()` con tu fuente

2. **Explorá Features Avanzadas**
   - SCD Type 2 para tracking de historia
   - Manejo de evolución de schema
   - Expectativas de calidad de datos con `@dlt.expect()`

3. **Lee Más**
   - [Blog Post de AutoCDC](https://www.databricks.com/blog/stop-hand-coding-change-data-capture-pipelines)
   - [Documentación de Delta Live Tables](https://docs.databricks.com/delta-live-tables/)
   - [Guía de Arquitectura Medallion](https://www.databricks.com/glossary/medallion-architecture)

---

## 🎉 ¡Felicitaciones!

Completaste el tutorial de AutoCDC. Ahora entendés:
- Cómo AutoCDC automatiza patrones CDC
- Cómo implementar SCD Type 1 con código mínimo
- Cómo construir pipelines de datos siguiendo arquitectura Medallion
- Cómo usar Delta Live Tables para workloads CDC de producción

¡Happy data engineering! 🚀